In [ ]:
# importing libraries
import pandas as pd
import numpy as np

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=None):
    """Find repo root by walking upward until expected project markers are found."""
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for path in [current, *current.parents]:
        if (path / "config").exists() and (path / "notebooks").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Could not find repo root. Launch notebook from inside the repository root.")

REPO_ROOT = find_repo_root()

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PREPROCESSING_DATA_DIR = REPO_ROOT / "data" / "processed" / "preprocessing"
OBJ2_DATA_DIR = REPO_ROOT / "data" / "processed" / "objective2_demand"
OBJ2_OUTPUT_DIR = REPO_ROOT / "outputs" / "objective2_demand"
OBJ2_MODEL_DIR = OBJ2_OUTPUT_DIR / "models"
OBJ2_VALIDATION_DIR = OBJ2_OUTPUT_DIR / "validation"
CONFIG_DIR = REPO_ROOT / "config"
CALENDAR_DIR = CONFIG_DIR / "calendars"
LOOKUP_DIR = CONFIG_DIR / "lookups"

for path in [OBJ2_DATA_DIR, OBJ2_OUTPUT_DIR, OBJ2_MODEL_DIR, OBJ2_VALIDATION_DIR]:
    path.mkdir(parents=True, exist_ok=True)


In [ ]:
# Read input files
weather_hist = pd.read_parquet(PREPROCESSING_DATA_DIR / "weather_hist_daily.parquet")
weather_future = pd.read_parquet(PREPROCESSING_DATA_DIR / "weather_future_daily_ukcp18.parquet")
UKCP18_LOOKUP_PATH = LOOKUP_DIR / "ukcp18_member_lookup.csv"
if not UKCP18_LOOKUP_PATH.exists():
    UKCP18_LOOKUP_PATH = PREPROCESSING_DATA_DIR / "ukcp18_member_lookup.csv"

ukcp18_lookup = pd.read_csv(UKCP18_LOOKUP_PATH)


In [ ]:
weather_hist.dtypes

In [ ]:
weather_hist.shape

In [ ]:
weather_hist.head()

In [ ]:
weather_hist.isna().sum()

In [ ]:
weather_hist.duplicated().sum()


In [ ]:
weather_future.dtypes

In [ ]:
weather_future.shape

In [ ]:
weather_future.head()

In [ ]:
weather_future.isna().sum()

In [ ]:
weather_future.duplicated().sum()


In [ ]:
ukcp18_lookup.dtypes

In [ ]:
ukcp18_lookup.shape

In [ ]:
ukcp18_lookup.head()

In [ ]:
ukcp18_lookup.isna().sum()

In [ ]:
ukcp18_lookup.duplicated().sum()


In [ ]:
# Check climate members
print("\nNumber of unique climate members in weather_future:", weather_future["climate_member"].nunique())
print("Number of unique climate members in lookup:", ukcp18_lookup["climate_member"].nunique())

print("\nSample climate members from weather_future:")
print(sorted(weather_future["climate_member"].dropna().unique())[:10])

print("\nSample climate members from lookup:")
print(sorted(ukcp18_lookup["climate_member"].dropna().unique())[:10])

In [ ]:
# Check date ranges
print("Historic weather min date:", weather_hist["date"].min())
print("Historic weather max date:", weather_hist["date"].max())
print("Historic weather unique dates:", weather_hist["date"].nunique())

print("\nFuture weather min date:", weather_future["date"].min())
print("Future weather max date:", weather_future["date"].max())
print("Future weather unique dates:", weather_future["date"].nunique())

In [ ]:
# Check missing dates for historic weather
hist_full_range = pd.date_range(weather_hist["date"].min(), weather_hist["date"].max(), freq="D")
hist_missing_dates = hist_full_range.difference(weather_hist["date"].drop_duplicates())

print("Historic weather missing dates count:", len(hist_missing_dates))
print("Historic weather missing dates sample:", hist_missing_dates[:10])

# Check missing dates for future weather
future_full_range = pd.date_range(weather_future["date"].min(), weather_future["date"].max(), freq="D")
future_missing_dates = future_full_range.difference(weather_future["date"].drop_duplicates())

print("\nFuture weather missing dates count:", len(future_missing_dates))
print("Future weather missing dates sample:", future_missing_dates[:10])

In [ ]:
# Check how many regions exist per date in historic weather
hist_region_count = weather_hist.groupby("date")["region"].nunique()

print("Historic weather regions per date:")
print(hist_region_count.value_counts())

print("\nHistoric dates where region count is not 2:")
print(hist_region_count[hist_region_count != 2])

In [ ]:
# Check how many region rows exist per date + climate_member in future weather
future_region_count = weather_future.groupby(["date", "climate_member"])["region"].nunique()

print("Future weather regions per date + climate_member:")
print(future_region_count.value_counts())

print("\nFuture date + climate_member where region count is not 2:")
print(future_region_count[future_region_count != 2])

In [ ]:
# Check unique regions
print("Historic regions:", sorted(weather_hist["region"].dropna().unique()))
print("Future regions:", sorted(weather_future["region"].dropna().unique()))

In [ ]:
# Quick numeric summary for weather variables
print("Historic weather summary:")
print(weather_hist[["tasmin_c", "tasmax_c", "tmean_c"]].describe().T)

print("\nFuture weather summary:")
print(weather_future[["tasmin_c", "tasmax_c", "tmean_c"]].describe().T)

## RESHAPING HISTORIC WEATHER ##

In [ ]:
# Reshape historic weather from long to wide
# One row per date, with separate columns for each region

weather_hist_wide = weather_hist.pivot(
    index="date",
    columns="region",
    values=["tasmin_c", "tasmax_c", "tmean_c"]
)

# Flatten multi-index column names
weather_hist_wide.columns = [
    f"{var}_{region}_c".replace("_c_", "_")
    for var, region in weather_hist_wide.columns
]

# Bring date back as a normal column
weather_hist_wide = weather_hist_wide.reset_index()

# Rename columns to required final format
weather_hist_wide = weather_hist_wide.rename(
    columns={
        "tasmin_c_eng_wales": "tasmin_eng_wales_c",
        "tasmax_c_eng_wales": "tasmax_eng_wales_c",
        "tmean_c_eng_wales": "tmean_eng_wales_c",
        "tasmin_c_scotland": "tasmin_scotland_c",
        "tasmax_c_scotland": "tasmax_scotland_c",
        "tmean_c_scotland": "tmean_scotland_c",
    }
)

weather_hist_wide.head()

## POPULATION WEIGHTED TEMPERATURE: HISTORIC WEATHER ##

In [ ]:
# Population-weighted GB temperature assumptions
#  Based on official mid-2024 population estimates

# England + Wales combined population
pop_eng_wales = 61_806_682

# Scotland population
pop_scotland = 5_546_900

# GB population total
pop_gb = pop_eng_wales + pop_scotland

# Calculate population weights
w_eng_wales = pop_eng_wales / pop_gb
w_scotland = pop_scotland / pop_gb

print("GB population total:", pop_gb)
print("England & Wales weight:", round(w_eng_wales, 6))
print("Scotland weight:", round(w_scotland, 6))
print("Weight sum:", w_eng_wales + w_scotland)

In [ ]:
# Create historic GB-weighted temperature features

weather_hist_features_daily = weather_hist_wide[["date"]].copy()

# Population-weighted minimum temperature
weather_hist_features_daily["tasmin_gb_c"] = (
    w_eng_wales * weather_hist_wide["tasmin_eng_wales_c"] +
    w_scotland * weather_hist_wide["tasmin_scotland_c"]
)

# Population-weighted maximum temperature
weather_hist_features_daily["tasmax_gb_c"] = (
    w_eng_wales * weather_hist_wide["tasmax_eng_wales_c"] +
    w_scotland * weather_hist_wide["tasmax_scotland_c"]
)

# Population-weighted mean temperature
weather_hist_features_daily["tmean_gb_c"] = (
    w_eng_wales * weather_hist_wide["tmean_eng_wales_c"] +
    w_scotland * weather_hist_wide["tmean_scotland_c"]
)

weather_hist_features_daily.head()

In [ ]:
# Check historic feature output
print(weather_hist_features_daily.shape)
print(weather_hist_features_daily.dtypes)
print(weather_hist_features_daily.isna().sum())
print(weather_hist_features_daily.duplicated(subset=["date"]).sum())

print("Min date:", weather_hist_features_daily["date"].min())
print("Max date:", weather_hist_features_daily["date"].max())
print("Unique dates:", weather_hist_features_daily["date"].nunique())

## RESHAPING FUTURE WEATHER ##

In [ ]:
# Reshape future weather from long to wide
# One row per date + climate_member

weather_future_wide = weather_future.pivot(
    index=["date", "climate_member"],
    columns="region",
    values=["tasmin_c", "tasmax_c", "tmean_c"]
)

# Flatten multi-index column names
weather_future_wide.columns = [
    f"{var}_{region}_c".replace("_c_", "_")
    for var, region in weather_future_wide.columns
]

# Bring date and climate_member back as columns
weather_future_wide = weather_future_wide.reset_index()

# Rename columns to required final format
weather_future_wide = weather_future_wide.rename(
    columns={
        "tasmin_c_eng_wales": "tasmin_eng_wales_c",
        "tasmax_c_eng_wales": "tasmax_eng_wales_c",
        "tmean_c_eng_wales": "tmean_eng_wales_c",
        "tasmin_c_scotland": "tasmin_scotland_c",
        "tasmax_c_scotland": "tasmax_scotland_c",
        "tmean_c_scotland": "tmean_scotland_c",
    }
)

weather_future_wide.head()

## POPULATION WEIGHTED TEMPERATURE: FUTURE WEATHER ##

In [ ]:
# Create future GB-weighted temperature features

weather_future_features_daily = weather_future_wide[["date", "climate_member"]].copy()

# Population-weighted minimum temperature
weather_future_features_daily["tasmin_gb_c"] = (
    w_eng_wales * weather_future_wide["tasmin_eng_wales_c"] +
    w_scotland * weather_future_wide["tasmin_scotland_c"]
)

# Population-weighted maximum temperature
weather_future_features_daily["tasmax_gb_c"] = (
    w_eng_wales * weather_future_wide["tasmax_eng_wales_c"] +
    w_scotland * weather_future_wide["tasmax_scotland_c"]
)

# Population-weighted mean temperature
weather_future_features_daily["tmean_gb_c"] = (
    w_eng_wales * weather_future_wide["tmean_eng_wales_c"] +
    w_scotland * weather_future_wide["tmean_scotland_c"]
)

weather_future_features_daily.head()

In [ ]:
# Check future feature output
print(weather_future_features_daily.shape)
print(weather_future_features_daily.dtypes)
print(weather_future_features_daily.isna().sum())
print(weather_future_features_daily.duplicated(subset=["date", "climate_member"]).sum())

print("Min date:", weather_future_features_daily["date"].min())
print("Max date:", weather_future_features_daily["date"].max())
print("Unique dates:", weather_future_features_daily["date"].nunique())
print("Unique climate members:", weather_future_features_daily["climate_member"].nunique())

## HDD and CDD ##

In [ ]:
# Define HDD and CDD base temperatures
# These are modelling assumptions for Task 2

# HDD base temperature:
# If GB mean temperature is below this threshold,
# heating-related demand is assumed to increase
hdd_base_c = 15.5

# CDD base temperature:
# If GB mean temperature is above this threshold,
# cooling-related demand is assumed to increase
cdd_base_c = 22.0

In [ ]:
# Create HDD and CDD for historic weather features

# HDD = max(hdd_base_c - tmean_gb_c, 0)
# Measures how much colder the day is than the heating base
weather_hist_features_daily["hdd"] = (
    hdd_base_c - weather_hist_features_daily["tmean_gb_c"]
).clip(lower=0)

# CDD = max(tmean_gb_c - cdd_base_c, 0)
# Measures how much hotter the day is than the cooling base
weather_hist_features_daily["cdd"] = (
    weather_hist_features_daily["tmean_gb_c"] - cdd_base_c
).clip(lower=0)

weather_hist_features_daily.head()

In [ ]:
# Create HDD and CDD for future weather features

# HDD = max(hdd_base_c - tmean_gb_c, 0)
weather_future_features_daily["hdd"] = (
    hdd_base_c - weather_future_features_daily["tmean_gb_c"]
).clip(lower=0)

# CDD = max(tmean_gb_c - cdd_base_c, 0)
weather_future_features_daily["cdd"] = (
    weather_future_features_daily["tmean_gb_c"] - cdd_base_c
).clip(lower=0)

weather_future_features_daily.head()

In [ ]:
# QA checks for HDD and CDD

# Check summary statistics
print("Historic HDD/CDD summary:")
print(weather_hist_features_daily[["hdd", "cdd"]].describe().T)

print("\nFuture HDD/CDD summary:")
print(weather_future_features_daily[["hdd", "cdd"]].describe().T)

# Check minimum values to confirm no negatives
print("\nHistoric HDD min:", weather_hist_features_daily["hdd"].min())
print("Historic CDD min:", weather_hist_features_daily["cdd"].min())

print("\nFuture HDD min:", weather_future_features_daily["hdd"].min())
print("Future CDD min:", weather_future_features_daily["cdd"].min())

In [ ]:
# Final column order for Task 2 outputs

weather_hist_features_daily = weather_hist_features_daily[
    ["date", "tasmin_gb_c", "tasmax_gb_c", "tmean_gb_c", "hdd", "cdd"]
].copy()

weather_future_features_daily = weather_future_features_daily[
    ["date", "climate_member", "tasmin_gb_c", "tasmax_gb_c", "tmean_gb_c", "hdd", "cdd"]
].copy()

print(weather_hist_features_daily.head())
print(weather_future_features_daily.head())

In [ ]:
# Save Task 2 output files

weather_hist_features_daily.to_parquet(OBJ2_DATA_DIR / "weather_hist_features_daily.parquet", index=False)
weather_future_features_daily.to_parquet(OBJ2_DATA_DIR / "weather_future_features_daily.parquet", index=False)

# Optional CSV copies for easier checking
weather_hist_features_daily.to_csv(OBJ2_VALIDATION_DIR / "weather_hist_features_daily.csv", index=False)
weather_future_features_daily.to_csv(OBJ2_VALIDATION_DIR / "weather_future_features_daily.csv", index=False)

print("Saved files:")
print("- weather_hist_features_daily.parquet")
print("- weather_future_features_daily.parquet")
print("- weather_hist_features_daily.csv")
print("- weather_future_features_daily.csv")


In [ ]:
readme_text = """
# weather_features_README

## Purpose
This README documents the creation of the Task 2 weather feature outputs for Objective 2.

---

## Task 2 objective
Task 2 creates climate features that can later be used in the daily climate-sensitive demand model.

Specifically, this task produces:

- GB-weighted daily minimum temperature
- GB-weighted daily maximum temperature
- GB-weighted daily mean temperature
- Heating Degree Days (`hdd`)
- Cooling Degree Days (`cdd`)

These features are created for both:

- historic weather data
- future UKCP18 weather data

## Final Dataset size

weather_hist_features_daily shape: (5479, 6)
weather_future_features_daily shape: (175320, 7)

---

## Why these weather features are needed

The final modelling target in Objective 2 is a single GB-level daily demand series, rather than separate regional demand series.

However, the weather inputs are available at regional level:
- England and Wales
- Scotland

This creates a mismatch:
- the response variable is GB-level
- the raw weather variables are regional

Therefore, a GB-level weather representation is required so that the climate features are aligned with the demand target.

A single GB-weighted temperature series is also needed so that:
- one consistent national daily weather signal is used in the model
- HDD and CDD can be calculated from a common GB temperature basis
- the same feature engineering logic can be applied to both historic and future climate data
- future climate members can later be compared consistently

---

### Data quality checks
The following checks were carried out:
- no unexpected missing values in weather variables
- no duplicate `date + region` rows in historic weather
- no duplicate `date + climate_member + region` rows in future weather
- expected regional values present
- date coverage checked
- climate member values inspected against lookup file

---

## Modelling assumptions

This task requires explicit modelling assumptions for:

1. GB temperature weighting
2. HDD base temperature
3. CDD base temperature

These assumptions are documented below.

---

## Assumption 1: GB-weighted temperature

### Why weighting is required
The model target is national GB daily electricity demand, but the available weather data is regional.

To create climate features that are aligned with the target variable, the regional temperatures must be combined into a single GB-level temperature representation.

### Weighting approach chosen
A **population-weighted** approach was used to combine England/Wales and Scotland temperature series.

### Why population-weighting was chosen
A demand-weighted approach would be conceptually strongest, because the model predicts aggregate GB electricity demand.

However, the available project datasets contain:
- GB-level demand
- regional weather

and do not contain:
- regional electricity demand shares for England/Wales and Scotland

Therefore, direct demand-weighting could not be estimated from the available data.

Population-weighting was chosen as a transparent and defensible proxy because:
- electricity demand is linked to where people and occupied buildings are concentrated
- it is more appropriate than equal weighting for a national demand model
- it is more relevant than land-area weighting for a demand-driven problem
- it produces a reproducible GB-level exposure measure

### Population-weight formula
Let:

- `pop_eng_wales` = England and Wales population
- `pop_scotland` = Scotland population

Then:

- `w_eng_wales = pop_eng_wales / (pop_eng_wales + pop_scotland)`
- `w_scotland = pop_scotland / (pop_eng_wales + pop_scotland)`

### Temperature aggregation formulas
The same weighting logic was applied to daily minimum, maximum, and mean temperature.

#### GB-weighted daily mean temperature
`tmean_gb_c = w_eng_wales * tmean_eng_wales_c + w_scotland * tmean_scotland_c`

#### GB-weighted daily minimum temperature
`tasmin_gb_c = w_eng_wales * tasmin_eng_wales_c + w_scotland * tasmin_scotland_c`

#### GB-weighted daily maximum temperature
`tasmax_gb_c = w_eng_wales * tasmax_eng_wales_c + w_scotland * tasmax_scotland_c`

### Recommendation for documentation
The exact population values and final calculated weights used in code should be recorded in the final project submission or appendix.

---

## Assumption 2: Heating Degree Days (HDD)

### What HDD represents
Heating Degree Days measure how much colder a day is than a chosen heating threshold.

This is used to capture the idea that colder days are associated with stronger heating-related electricity demand effects.

### Why HDD is needed
A single temperature variable may not fully capture non-linear temperature-demand behaviour.

In practice, electricity demand often becomes more sensitive when temperatures fall below a certain threshold.

HDD is therefore used to represent cold-weather demand pressure in an interpretable way.

### HDD formula
`hdd = max(hdd_base_c - tmean_gb_c, 0)`

### HDD base temperature used
- `hdd_base_c = 15.5`

### Why 15.5°C was used
This was selected as a practical initial modelling threshold for heating sensitivity.

Reasoning:
- it is a defensible energy-modelling threshold
- it allows cold-weather demand effects to be represented without treating all mild days as strong heating days
- it provides a transparent starting point for feature engineering
- it can later be validated during model training in Task 3

This should be interpreted as a modelling assumption, not as a fixed physical truth.

---

## Assumption 3: Cooling Degree Days (CDD)

### What CDD represents
Cooling Degree Days measure how much warmer a day is than a chosen cooling threshold.

This is used to capture the possibility that hotter days may increase cooling-related electricity demand.

### Why CDD is needed
Although cooling sensitivity is generally weaker than heating sensitivity in GB, future climate analysis may make warm-weather effects more relevant.

Including CDD ensures that the feature set can represent both:
- cold-weather demand effects
- hot-weather demand effects

### CDD formula
`cdd = max(tmean_gb_c - cdd_base_c, 0)`

### CDD base temperature used
- `cdd_base_c = 22.0`

### Why 22.0°C was used
This was selected as a conservative initial threshold for cooling sensitivity.

Reasoning:
- it avoids overstating cooling effects on moderately warm days
- it is appropriate for a GB context where widespread cooling demand is weaker than heating demand
- it supports future climate scenario modelling without forcing a strong cooling response on mild temperatures
- it can later be reviewed during model validation in Task 3

This should again be interpreted as an initial modelling assumption.

## Note on `.clip(lower=0)`

In the Python implementation, HDD and CDD were created using `.clip(lower=0)`.

This sets any value below 0 to 0, while leaving positive values unchanged.

### Why it is needed for HDD and CDD
The HDD and CDD formulas are:

- `hdd = max(hdd_base_c - tmean_gb_c, 0)`
- `cdd = max(tmean_gb_c - cdd_base_c, 0)`

This means:
- HDD should never be negative
- CDD should never be negative

If the raw subtraction produces a negative value, that means the threshold has not been exceeded, so the correct degree-day value is 0.


---

## Limitations and interpretation

### 1. Population-weighting is a proxy
Population-weighting is not identical to demand-weighting.

It is used because regional demand-share data was not available.

### 2. HDD and CDD thresholds are modelling assumptions
The chosen base temperatures are initial, theory-informed thresholds rather than final confirmed optima.

Their usefulness should be assessed in Task 3 when the demand model is trained and validated.

### 3. Single national temperature representation
Using one GB-weighted temperature simplifies the national demand modelling problem.

This is appropriate for the current project design, but it does not explicitly model separate regional demand responses.


---


"""

# Save README as markdown
readme_path = OBJ2_VALIDATION_DIR / "weather_features_README.md"
readme_path.parent.mkdir(parents=True, exist_ok=True)
readme_path.write_text(readme_text, encoding="utf-8")

print(f"Saved file: {readme_path}")

In [ ]:
print("weather_hist_features_daily shape:", weather_hist_features_daily.shape)
print("weather_future_features_daily shape:", weather_future_features_daily.shape)
